<a href="https://colab.research.google.com/github/ds-20195/notebooks/blob/main/drafts/lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data and the State
**DATA 21905/31905**

# Week 1: 3/24 - 3/27

# Lab 1: Introduction to Python and Census

## Resources




* Python
  * [DataCamp's interactive Intro to Python tutorial](https://www.learnpython.org/)
  * [Python for Everybody (University of Michigan MOOC)](https://www.py4e.com/lessons)
  * [A short Python and NumPy tutorial (Stanford CS231)](http://cs231n.github.io/python-numpy-tutorial/)
  * [Official Python tutorial](https://docs.python.org/3/tutorial/index.html)
  * [Jake VanderPlas' _A Whirlwind Tour of Python_](https://github.com/jakevdp/WhirlwindTourOfPython/tree/master?tab=readme-ov-file) (particularly chapters 1–8)
  * [Cornell CS 1110 lecture videos (Fall 2020)](https://vod.video.cornell.edu/channel/CS+1110+Fall+2020/179890731)
* Pandas
    * [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html)
    * [Official pandas tutorials](https://pandas.pydata.org/docs/getting_started/index.html#intro-to-pandas)
    * [Data Manipulation with Pandas (chapter 3 of VanderPlas' *Python Data Science Handbook*)](https://jakevdp.github.io/PythonDataScienceHandbook/03.00-introduction-to-pandas.html)
    * [Introduction to GeoPandas](https://github.com/geopandas/geopandas/blob/main/doc/source/getting_started/introduction.ipynb) (Jupyter notebook)
    * [Comparison with R / R libraries](https://pandas.pydata.org/docs/getting_started/comparison/comparison_with_r.html) (a handy Rosetta stone if you've previously used R!)
    * [UC Berkeley Data 100 Early Lectures on Pandas and Visualization](https://ds100.org/su25/)
* Matplotlib
  * [Official Pyplot tutorial (short)](https://matplotlib.org/tutorials/introductory/pyplot.html)
  * [Visualization with Matplotlib (chapter 4 of VanderPlas' *Python Data Science Handbook*)](https://jakevdp.github.io/PythonDataScienceHandbook/04.00-introduction-to-matplotlib.html)
  * [Gallery of Matplotlib examples](https://matplotlib.org/gallery/index.html)

   

## Exploring the US Census

In lab section, we'll explore the most famous and essential dataset published by the U.S. Census Bureau: the decennial enumeration of the U.S. population. This enumeration is [required by Article I of the U.S. Constitution](https://www.census.gov/programs-surveys/decennial-census/about/census-constitution.html), and the challenges of processing Census data as the national population grew throughout the 19th century led to the development of [a tabulating machine](https://www.computerhistory.org/revolution/punched-cards/2/2) that was [first used for processing the 1890 U.S. Census](https://www.smithsonianmag.com/smithsonian-institution/herman-holleriths-tabulating-machine-2504989/). This machine read punched cards that encoded Census responses and generated summary statistics (such as the number of foreign-born children under 18 in a particular state, or the number of married women in a particular city). The company that developed this tabulating machine was later renamed to IBM.

The technical demands of analyzing U.S. Census data were therefore a catalyst in the early development of computing; in the 130 years or so since the dawn of this automated tabulation technology, Census data has become exponentially more detailed and accessible. Here, we'll focus on a small slice of the 2010 and 2020 decennial Census data published online via the [Census API](https://www.census.gov/data/developers/data-sets.html).

First we need to do some package setups. Run the follwing cells.

It is not important that you understand the technical details of package set up, but feel free to read the comments below to get a basic idea.

In [ ]:
# You can ignore the contents of this cell for now. This is just to make sure we have the correct
# packages are installed for this lab.

# The `census` package is a wrapper around the US Census API, which allows us to easily query census
# data from Python easily. The `us` package provides a simple interface to get information about
# US states, such as their names and abbreviations.
!pip install -q census us

In [ ]:
# Here we set up our working environment by importing the necessary packages, classes, and
# functions.

# We import the `pandas` package, which is a powerful data manipulation library in Python.
# It provides data structures like DataFrames that make it easy to work with structured data
# (more on this later).
import pandas as pd


# From the `census` package, we import the `Census` class, which allows us to interact with the US
# Census API an ergonomic way. We will use this class to query census data in this lab.
from census import Census

# From the `us` package, we import the `states` module, which provides information about US states,
# such as their names, abbreviations, and FIPS codes. This will be useful for working with census
# data, which often uses FIPS (Federal Information Processing Standards) codes to identify states.
from us import states

## Fetching population data

Here, we'll retrieve population statistics by race from the Census API for all counties in Illinois. Census data releases are organized into tables; [Table P1](https://data.census.gov/table/DECENNIALPL2020.P1?g=040XX00US36) breaks down population by racial category.

The official column names used by the Census Bureau are frequently inscrutable (for instance, the `P1_004N` column corresponds to Black population), and so we provide a dictionary below that we will later use to make working with the data more intuitive. (Full column definitions are available in the [API documentation](https://api.census.gov/data/2020/dec/pl/variables.html).)

In [ ]:
p1_population_columns = {
    "P1_003N": "white",  # White alone
    "P1_004N": "black",  # Black or African American alone
    "P1_005N": "amin",  # American Indian and Alaska Native alone
    "P1_006N": "asian",  # Asian alone
    "P1_007N": "nhpi",  # Native Hawaiian and Other Pacific Islander alone
    "P1_008N": "other",  # Some Other Race alone
    "P1_009N": "two_or_more",  # Two or more races
}

When fetching data from the Census API, we usually identify geographic regions (states, counties, etc.) by their [FIPS code](https://en.wikipedia.org/wiki/Federal_Information_Processing_Standard_state_code). For instance, the FIPS code for the state of Illinois is 17. The `us` package exposes identifiers (FIPS codes, postal abbreviations) for the U.S. states.

In [ ]:
state = states.IL
print(state, state.abbr, state.fips)

Now we will make use of the `Census` class we imported from the `census` module and specify:
* The dataset we wish to query (in this case the `pl` dataset which stands for "Public Law 94-171")
* The columns we want to fetch from the chosen dataset (the `NAME` column, and the 7 columns from Table P1 listed in `p1_population_columns` above).
* The regions we want to fetch column values for (all counties in Illinois).

>Coding notes:
> * The syntax `f"state:{state.fips}"` is a kind of [string formatting](https://docs.python.org/3/tutorial/inputoutput.html) in Python, which allows you to dynamically change the content of the string based on other defined variables (in this case, `state.fips`). You can alternatively replace this code with the hard-coded FIPS values (e.g., `"state:01"`).
> * The asterisk `*` allows you to select *all* values corresponding to either counties or states.

In [ ]:
census = Census(
    key="",      # We do not have a Census API key, so we leave this blank.
    year=2020    # We specify that we would like to use the 2020 Census data.
)

census_column_list = ["NAME"] + list(p1_population_columns.keys())
# The `NAME` column contains the name of the geographic area (in this case, the county name),
# and the keys of our `p1_population_columns` dictionary correspond were the columns P1_003N to
# P1_009N in the census data, which contain the population counts for different racial groups.

county_populations = census.pl.get(
    census_column_list,
    geo={
        "for": "county:*",
        "in": f"state:{state.fips}",
    },
)

**Note: there are several ways to modify the above code to get data for *multiple* counties and *multiple* states.**

For example, if you want to get data for multiple counties you can simply list them out by their FIPS codes.

```python
# This is not an executable cell
county_populations = census.pl.get(
    census_column_list,
    geo={
        "for": "county:001,002,003",
        "in": f"state:{state.fips}",
    }
)
```

Or, if you want to get data for all counties in *multiple states* you can list out the states by their FIPS codes.

```python
# This is not an executable cell
county_populations = census.pl.get(
    census_column_list,
    geo={
        "for": "county:*",
        "in": "state:01,02",
    }
)
```

The Census API returns a list of dictionaries; each contains column values for a county which we can print out.

In [ ]:
# There are 102 counties in Illinois, so we should have 102 rows in our list of dictionaries
# `county_populations`, but we are just going to print out the first 3 rows to check that our
# data looks correct.
county_populations[:3]

However, a list of dictionaries is not an ideal way of interacting with large datasets (unless you reall, really like for loops), so we will now make use of the data manipulation library `pandas`.

We can convert this raw response to a Pandas `DataFrame` without any further manipulation. For now, you can simply think of Pandas DataFrames as Python's version of spreadsheets.

In [ ]:
# We create the dataframe `race_df` from the list of dictionaries `county_populations`.
# Each dictionary in the list represents a row in the dataframe, where the keys of the dictionary
# correspond to the column names and the values correspond to the data in those columns.
#
# We also make sure to store the dataframe in a variable called `race_df`, which so we can extract
# data from it later on in the lab.
race_df = pd.DataFrame(county_populations)

# We will have colab print out the dataframe `race_df` so we can see what it looks like.
race_df

Okay, that's a lot of data! Generally, we don't want to view our entire Dataframe, so we can instead use the `head(<number_of_rows.>)` and `tail(<number_of_rows>)` methods to take a peak at the first or last few rows of the DataFrame.

In [ ]:
race_df.head(3)

## Data Querying and Postprocessing

### Modifying Dataframe Column Names and Indices

Currently, our `race_df` dataframe is a bit hard to work with (what does "P1_005N" mean again?), but we can make `race_df` more usable by applying our mapping from raw column names to human-readable racial categories. Fortunately, pandas DataFrames provide us with a convenient way of doing this: the `rename` method.

Below is an example of how to use the `rename` method to change the column titled "state" to "state_fips_code".

In [ ]:
race_df = race_df.rename(columns= {"state": "state_fips", "county": "county_fips"})
race_df

#### TASK 1

Recall that we made a dictionary above called `p1_population_columns`. This maps census column names to namess that can be more easily read and remembered. use the `rename` method to introduce human-readable names into the `race_df` dataframe. It's possible to do this by renaming teh columns individually like we did in the previous cell, but you can accomplish the same thing more simply by using the dictionary.


In [ ]:
# YOUR CODE GOES HERE

# but raise your hand and ask for help if you get stuck!
race_df # This just prints out the dataframe so you can see if your code worked correctly.

In [ ]:
# DELETE ME!!!!!!!!
race_df = race_df.rename(columns=p1_population_columns)
race_df

Since we are at the University of Chicago, we might be interested in looking at the data for Cook county, which just so happens to be at index 15 of the dataframe as we can see below:

In [ ]:
race_df.head(20)

And we can grab row containing just the data for Cook County using the `loc[<index>]` method for pandas dataframes. Here we can see that the index for Cook County is 15.

In [ ]:
race_df.loc[15]

Locaing data by it's integer index can still be troublesome, so it is often convenient to _reindex_ the dataframe to use some other unique identifier.

In [ ]:
# We will now make it so that the county names are the index of our dataframe.
race_df = race_df.set_index("NAME")
race_df

Notice that the leftmost column has now changed to "NAME" -- this means that the "NAME" column has now become the index of our dataframe, and we can access rows by their name

In [ ]:
# Remember that the `loc` method accesses the INDEX of the dataframe
race_df.loc["Cook County, Illinois"]

Note that if we now try to access the data for Cook County using it's integer index, we will get a `KeyError`

In [ ]:
race_df.loc[15]

One more thing. Since we are looking at racial composition data, we probably don't need to hold onto state or county fips codes in our dataframe any more, so let's use the `drop` method to remove them.

In [ ]:
race_df = race_df.drop(columns=["state_fips", "county_fips"])
race_df

### Data Manipulation

Pandas dataframes come equiped with a load of convenience functions that allow us to manipulate and process data. For example, we can sum the values in the "white" column to obtain the total number of people that identify as White in the state

In [ ]:
# Here, the syntax `race_df["white"]` accesses the "white" column of the dataframe, which contains
# the population counts of white people in each county. The `sum()` method then adds up all the
# values in that column to give us the total population of white people across all counties in Illinois.
race_df["white"].sum()

Or we can sum the values in every column at once to compute population totals by racial category for the state.

In [ ]:
race_df.sum(axis=0)  # axis = 0 means sum across columns

Similarly, we can sum the values in each row to compute population totals (across all racial categories) for each county.

In [ ]:
race_df.sum(axis=1)  # axis = 1 means sum across rows

In order to answer questions like "which county in Illinois has the largest _share_ of Black population?", it's necessary to convert absolute population counts to percentages. To do this, it's convenient to first compute a `total` column and add that to the dataframe.

In [ ]:
# Here the syntax race_df["total"] creates a new column in the dataframe called "total", and assigns
# to it the result of summing all the values in each row of the dataframe (i.e. the total
# population of each county across all racial groups).
race_df["total"] = race_df.sum(axis=1)

In [ ]:
race_df.head(5)

To create the percentage column for a particular racial category, we can then use _vector arithmetic_ to divide a racial column by the total population:

In [ ]:
# Here, we create a new column called "white_pct" that contains the percentage of the population that is
# white in each county. We calculate this by dividing the total population of each county (which we
# just calculated in the "total" column) by the population of white people in each county
# (which is in the "white" column), and then multiplying by 100 to get a percentage.
race_df["white_pct"] = 100 * (race_df["total"] / race_df["white"])
race_df.head(5)

And if we want to check our work, we can extract the columns that we used in our computations from `race_df` and store those in a new dataframe.

In [ ]:
# Note that the syntax `race_df[["white", "total", "white_pct"]]` creates a new dataframe that only
# contains the "white", "total", and "white_pct" columns from the original dataframe `race_df`.
computation_check_df = race_df[["white", "total", "white_pct"]]
computation_check_df.head(5)

#### TASK 2

Add a column called "black_pct" to `race_df` that records the percent of the population that identifies as Black.

In [ ]:
# YOUR CODE GOES HERE

# but raise your hand and ask for help if you get stuck!
race_df[["black", "total", "black_pct"]] # This just prints out the dataframe so you can see if your code worked correctly.

We would like to have a column containing the racial population percentage for each racial category, which we could do by hand, but is easier to do with a `for` loop.

In [ ]:
# Here we are just extracting the population column names from the `p1_population_columns`
# dictionary and storing them in a list called `categories`.
categories = list(p1_population_columns.values())
categories

In [ ]:
# Now we loop through the population categories in our list and add a column to our dataframe for each.
for col in categories:
    race_df[f"{col}_pct"] = 100 * race_df[col] / race_df["total"]

race_df.head(3)

Armed with these percentages, we can now answer demographic questions cast in terms of population shares. For instance, sorting `race_df` by `white_pct` shows that Jasper County contains the largest share of white population in the state, while Cook County contains the smallest share.

In [ ]:
# Note that the `sort_values` method sorts values from smallest to largest by default, so the
# county with the county with the highest percentage of white people will be at the bottom.
race_df = race_df.sort_values(by=["white_pct"])
race_df

#### TASK 3

The `sort_values` method has a keyword parameter called "ascending" determines if the values in the dataframe are sorted from smallest-to-largest or vice versa (c.f. [the documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html]).


Modify the following code cell to rank counties in Illinois by `black_pct` showing counties with the largest percentage at the top.

In [ ]:
# Modify this!
race_df = race_df.sort_values(by=["white_pct"], ascending=True)
race_df

### Filtering

One last thing that we can do in pandas is _filter_ our dataframe. Suppose, for example, we wanted to know which counties were more than 15% Black. To do this, we must first construct a _boolean mask_ for our dataframe

In [ ]:
# This will print out a boolean series that indicates whether the population in a given county is
# more than 15% black.
race_df["black_pct"] > 15

iYou'll notice that the output above is a sequence of boolean (True or False) values. To actually get the rows in the dataframe that satisfy our condition, we can use the following syntax:

In [ ]:
# This syntax says "access those locations (indexex) in the dataframe where the condition
# `race_df["black_pct"] > 15` is true".
race_df.loc[race_df["black_pct"] > 15]

And we can make more complicated filters too!

In [ ]:
# We can also combine multiple conditions using the `&` operator, which stands for "and". This will give us
# the counties where the population is more than 15% black AND more than 70% white.
race_df.loc[(race_df["black_pct"] > 15) & (race_df["white_pct"] > 70)]

In [ ]:
# Likewise, the `|` operator, which stands for "or". This will give us
# the counties where the population is more than 20% black OR more than 5% asian.
race_df.loc[(race_df["black_pct"] > 20) | (race_df["asian_pct"] > 5)]

In [ ]:
# And vector operations still work as expected.
# This will give us the counties where the total black and asian population is more than 25% of the total population.
race_df.loc[(race_df["black_pct"] + race_df["asian_pct"]) > 25]

#### TASK 4

Use a boolean mask to find all of the counties where both `other_pct` and `two_or_more_pct` are larger than 10. Hint: there should be 4 counties!

In [ ]:
# Modify the line below
filtered_df = race_df

# raise your hand and ask for help if you get stuck!
filtered_df

## Plotting with Matplotlib

Matplotlib is another essential package in the Python data science ecosystem. Matplotlib can generate almost any 2D plot imaginable (see the [examples gallery](https://matplotlib.org/stable/gallery/index.html)), and it's tightly integrated with Pandas and GeoPandas.


### Importing
We conventionally load Matplotlib as `plt` for short. The examples here also use [NumPy](https://numpy.org/doc/stable/reference/) (imported as `np` for short) to generate large arrays of random numbers.



In [ ]:
import matplotlib.pyplot as plt
import numpy as np


# tells Google Colab to render all plots and charts as SVG (Scalable Vector Graphics) instead of
# the default PNG which will make our plots look nicer
%config InlineBackend.figure_formats = ["svg"]

### Line Plots

When working in a Jupyter notebook, it's often preferable to configure Matplotlib to emit [vector images](https://en.wikipedia.org/wiki/Vector_graphics). We're going to plot a simple line next, and save it as both a PNG and an SVG.  Navigate to the file button on the left hand tool menu to open up both and zoom way in to see the difference!

In [ ]:
# this plots percentage of population that is white on the x-axis and
# total population on the y-axis

white_pct = race_df["white_pct"]
total_pop = race_df["total"]

fig, ax = plt.subplots()
ax.plot(white_pct, total_pop)
ax.set_xlabel("Percentage of Population that is white")
ax.set_ylabel("Total Population")
ax.set_title("White Population Percent vs. Total Population")
plt.savefig("line.png") #this is how you save a plot
plt.savefig("line.svg")
plt.show()



### Scatter Plots

We can also plot the data as individual points instead of lines.

In [ ]:

fig, ax = plt.subplots()
ax.scatter(white_pct, total_pop)  # scatter instead of plot here
ax.set_xlabel("Percentage of Population that is white")
ax.set_ylabel("Total population")
plt.show()

### Mixing and Matching


We can also combine different plot types into one shared plot.



In [ ]:
fig, ax = plt.subplots()
ax.plot(white_pct, total_pop, label="Line", color="red")
ax.scatter(white_pct, total_pop, label="Scatter", color="blue")

ax.set_xlabel("Percentage of Population that is white")
ax.set_ylabel("Total population")
ax.legend(loc="upper right") # where to put the legend in the plot

plt.show()

#### TASK 5

Matplotlib's `scatter` function comes with a parameter `s` (short for "size") which controls the size of the dots. Modify the code below so that
- the scatter plot has dots of size 100
- the points of the scatter plot are colored "black".

In [ ]:
fig, ax = plt.subplots()
ax.plot(white_pct, total_pop, label="Line", color="red")

ax.scatter(white_pct, total_pop, label="Scatter", color="blue")

ax.set_xlabel("Percentage of Population that is white")
ax.set_ylabel("Total population")
ax.legend(loc="upper right")

plt.show()

### Histograms

In [ ]:
fig, ax = plt.subplots()
ax.hist(race_df["amin_pct"] , bins=10, density=True)
ax.set_title("Distribution of percentage of American Indian and \n Alaska Native alone in County populations")
ax.set_xlabel("Percent")
ax.set_ylabel("Density")
plt.show()

We can also create a histogram using Panda's `.hist` function, which wraps around matplotlib's histogram function.

In [ ]:
ax = race_df["black_pct"].hist(bins=20)
ax.set_xlabel("% Black")
ax.set_ylabel("Count")
ax.set_title("Black population in Illinois counties")
plt.show()

This histogram indicates that most counties in Illinois have a relatively small Black population share (most of the mass is below the 10% mark), but there are a few outlying counties with a large concentration of Black population.

### Visualizing demographics with a pie chart

We can visualize the demographic breakdown of a particular county with [Pandas' pie chart function](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.pie.html), which wraps around matplotlib's pie chart function.

In [ ]:
pie_df = race_df[p1_population_columns.values()] #selects only population counts for all races
county = "Cook County, Illinois"
ax = pie_df.T.plot.pie(y=county, figsize=(10, 10), legend=False)  # what does T do?
ax.set_title(f"Population of {county} by race")
ax.set_ylabel("")
plt.show()

We can visualize the demographic breakdown of the whole state in a similar fashion: first, we compute total population by race for the entire state (as illustrated in the section above), and then we plot the resulting Pandas `Series`. This pie chart improves on the county-level pie chart above by showing percentages for each racial category (rounded to the nearest 0.1%).

In [ ]:
ax = pie_df.sum(axis=0).plot.pie(figsize=(10, 10), autopct="%1.1f%%")
ax.set_ylabel("")
ax.set_title(f"Population of {state} (state) by race")
plt.show()

## LAB TURN IN PLOT

Go to the code block where we designated Illinois as the state and change it to New York. Then, restart the session and run all to re-produce the  plot directly above (non-hispanic population) for that state, make sure to uncomment the save figure line so you can export the plot and turn it in on gradescope.

## Adding a dimension: Hispanic or Latino origin

The U.S. Census Bureau does not consider "Hispanic or Latino" a racial category; [rather, it is an ethnic category](https://www.census.gov/acs/www/about/why-we-ask-each-question/ethnicity/) ([this may change with the 2030 Census](https://www.npr.org/2023/01/26/1151608403/mena-race-categories-us-census-middle-eastern-latino-hispanic)). Hispanic or Latino origin and racial identification were therefore considered in separate questions on the 2020 Census questionnaire (if you're curious about the precise wording of the questions, see [this PDF of the questionnaire](https://www2.census.gov/programs-surveys/decennial/2020/technical-documentation/questionnaires-and-instructions/questionnaires/2020-informational-questionnaire-english_DI-Q1.pdf)).

[Table P2](https://data.census.gov/table/DECENNIALPL2020.P2?g=040XX00US36) contains population totals broken down by both race _and_ Hispanic/Latino origin. Let's retrieve county-level statistics from this table. (As with Table P1, column definitions are available in the [API documentation](https://api.census.gov/data/2020/dec/pl/variables.html).) When we rename columns derived from Table P2, **we'll use the prefix `nh_` for columns containing non-Hispanic population and the prefix `h_` for columns containing Hispanic population.**

In [ ]:
p2_non_hispanic_population_columns = {
    "P2_005N": "nh_white",  # White alone
    "P2_006N": "nh_black",  # Black or African American alone
    "P2_007N": "nh_amin",  # American Indian and Alaska Native alone
    "P2_008N": "nh_asian",  # Asian alone
    "P2_009N": "nh_nhpi",  # Native Hawaiian and Other Pacific Islander alone
    "P2_010N": "nh_other",  # Some Other Race alone
    "P2_011N": "nh_two_or_more",  # Two or more races
}

In [ ]:
# Make them type this out
county_non_hispanic_populations = census.pl.get(
    ("NAME", *p2_non_hispanic_population_columns),
    geo={
        "for": "county:*",
        "in": f"state:{state.fips}",
    },
)

race_ethnicity_df = pd.DataFrame(county_non_hispanic_populations)
race_ethnicity_df.head(5)

We can apply the same postprocessing steps we applied to Table P1 to clean up the column names.

In [ ]:
Race_ethnicity_df = (
    race_ethnicity_df.rename(
        columns={"NAME": "name", **p2_non_hispanic_population_columns}
    )
    .drop(columns=["state", "county"])
    .set_index("name")
)

In [ ]:
race_ethnicity_df

Table P2 only includes _non_-Hispanic population by race, but we can easily derive Hispanic population by race by incorporating totals from Table P1.

In [ ]:
for category in categories:
    race_ethnicity_df[f"h_{category}"] = (
        race_df[category] - race_ethnicity_df[f"nh_{category}"]
    )

In [ ]:
race_ethnicity_df.head(5)

In [ ]:
race_ethnicity_df["total"] = race_ethnicity_df.sum(axis=1)

In [ ]:
non_hispanic_categories = list(p2_non_hispanic_population_columns.values())
non_hispanic_categories

For convenience, let's construct subtotals for the Hispanic and non-Hispanic populations.

In [ ]:
race_ethnicity_df["nh_total"] = race_ethnicity_df[non_hispanic_categories].sum(axis=1)

In [ ]:
hispanic_categories = [
    "h_white",
    "h_black",
    "h_asian",
    "h_nhpi",
    "h_other",
    "h_two_or_more",
]

In [ ]:
race_ethnicity_df["h_total"] = race_ethnicity_df[hispanic_categories].sum(axis=1)

In [ ]:
race_ethnicity_df.head(5)

Now that we've broken down the state population along two dimensions (race and Hispanic/Latino origin), we can answer some more sophisticated demographic questions: for instance, how does the distribution of racial identification differ between the Hispanic and non-Hispanic populations?

In [ ]:
ax = (
    race_ethnicity_df[["nh_total", "h_total"]]
    .rename(columns={"nh_total": "non-Hispanic", "h_total": "Hispanic"})
    .sum(axis=0)
    .plot.pie(figsize=(6, 6), autopct="%1.1f%%")
)
ax.set_ylabel("")
ax.set_title(f"Population of {state} (state) by Hispanic/Latino origin")
plt.show()

In [ ]:
ax = (
    race_ethnicity_df[hispanic_categories]
    .rename(
        columns={
            "h_other": "SomeOther",
            "h_asian": "Asian",
            "h_black": "Black",
            "h_white": "White",
            "h_two_or_more": "Multi",
            "h_amin": "AMIN",
            "h_nhpi": "NHPI",
        }
    )
    .sum(axis=0)
    .sort_values(ascending=False)
    .plot.bar(figsize=(6, 6), rot=0)
)
ax.set_title(f"Hispanic population of {state}")
plt.show()

In [ ]:
ax = (
    race_ethnicity_df[non_hispanic_categories]
    .rename(
        columns={
            "nh_other": "SomeOther",
            "nh_asian": "Asian",
            "nh_black": "Black",
            "nh_white": "White",
            "nh_two_or_more": "Multi",
            "nh_amin": "AMIN",
            "nh_nhpi": "NHPI",
        }
    )
    .sum(axis=0)
    .sort_values(ascending=False)
    .plot.bar(figsize=(6, 6), rot=0)
)
ax.set_title(f"non-Hispanic population of {state}")
plt.show()
#plt.savefig("lab_turn_in.svg")

## Homework 2, due Tuesday Feb 11, 1:25pm



**Warm up question:**  Among all counties in Illinois with no more than 10,000 people total, which county has the second largest percentage of Black population? Explain in simple English how you used the tools we learned to find this information.


For the **data product** part of your homework, explore the census race/ethnicity data.  Pose ONE question and show us a plot that answers it.  Here are some examples to get you thinking...
- How does the share of "white" people change in Texas if I only sort by race (ignoring ethnicity), or if I consider both race and ethnicity and treat Hispanic as a race?  (This could be a pair of pie charts side-by-side, for instance.)
- Which state had its number of multi-racial respondents jump most in absolute numbers between the 2010 and 2020 Census?
- What's the share of Hispanic respondents choosing "Some Other Race" in California? Is it very different from Florida?
- What's the Whitest state with at least 3 million people?
- Which state has the highest share of people who said they belong to ALL SIX races?  (note: to do this, you'd need to fetch new columns from the API)

Be creative!


**THIS NEEDS MOON TO UPDATE**
For the **reading response** part of your homework, refer to the chapter by Nobles.  Pick out and quote one juicy sentence that you underlined when you read the chapter.  React to it and connect it to something from either Scott or Hacking (two other authors we've been discussing).

In [ ]:

#HW CODE ANSWER --- DELETE BEFORE POST
race_df.loc[race_df["total"]<10000].sort_values(by="black_pct", ascending =False)